# Prompting & In-Context Learning: build an induction head, measure few-shot beating zero-shot

You have a capable model with **frozen weights** and a brand-new task. **In-context learning (ICL)**
is the ability to do that task from **examples in the prompt — with no gradient updates.** The
"learning" happens in the forward pass (activations), not the parameters.

This notebook makes ICL **concrete and measured** on a tiny transformer we build and train ourselves
— no pretrained weights, no internet, runs on CPU in seconds. Four experiments, each proving one
claim from the [concept page](../16-Prompting-and-In-Context-Learning.md):

1. **Induction head** — train a 2-layer attention-only model on a *look-back-and-copy* task, then
   read its attention and *see* it copy from context. This is the mechanism behind few-shot copying.
2. **Few-shot > zero-shot** — measure accuracy vs `k` in-context demonstrations. Zero-shot sits at
   chance; one demonstration switches the task on. No weight update.
3. **Sensitivity** — show the same demonstrations in a different order and watch the answer flip.
   ICL is powerful *and* brittle.
4. **Calibration** — measure a prompt's label bias with a content-free input and divide it out
   (Zhao et al. 2021); the content-free distribution flattens to uniform.

> Every number here is recomputed from the canonical script `prompting_icl.py`, so the notebook,
> the page, and the figures cannot drift. The run is **pinned to CPU** for reproducibility; we still
> detect and print the real accelerator honestly.

## Setup: device honesty and the imports

We import the canonical functions from `prompting_icl.py` (the same code the page and figures use).
The model is small, so the reproducible run is **pinned to CPU** — seeded float math drifts across
CPU/MPS/CUDA, and that would break the "page == notebook == .py" contract. We print the *detected*
accelerator honestly so the device line never lies.

In [1]:
import torch

import prompting_icl as icl  # the canonical demo (same code as the page and the figures)

print(f"device: {icl.DEVICE} (detected {icl.DETECTED_DEVICE}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)
print("vocab size:", icl.VOCAB_SIZE, "| symbols:", icl.N_SYMBOLS, "| seq len:", icl.SEQ_LEN)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0
vocab size: 41 | symbols: 40 | seq len: 14


## The task: in-context recall (the smallest task that *requires* reading the prompt)

Each sequence is a run of random `(key, value)` pairs, then a special `CUE` token, then a repeated
key. The model must predict the **value that followed that key earlier**. Crucially, the `key → value`
mapping is **resampled every sequence**, so the answer is *never* stored in the weights — it exists
only in the prompt. Predicting it forces the model to **look back and copy**: an induction operation.
Let's look at one sequence.

In [2]:
gen = torch.Generator().manual_seed(42)
tokens, targets = icl.make_batch(1, generator=gen)
ids = tokens[0].tolist()
print("sequence (key, value, key, value, ..., CUE, query_key):")
print("  ", ids)
print(f"  CUE token id = {icl.CUE_ID}  (marks 'the next token repeats an earlier key')")
print(f"  query key (last token) = {ids[-1]}")
print(f"  target the model must predict = {targets[0].item()} "
      f"(the value that followed key {ids[-1]} earlier)")
print("shape of a batch of tokens:", tuple(tokens.shape))

sequence (key, value, key, value, ..., CUE, query_key):
   [22, 10, 9, 26, 32, 22, 15, 17, 0, 39, 3, 27, 40, 3]
  CUE token id = 40  (marks 'the next token repeats an earlier key')
  query key (last token) = 3
  target the model must predict = 27 (the value that followed key 3 earlier)
shape of a batch of tokens: (1, 14)


## The model: a 2-layer attention-only transformer (the smallest thing that can grow an induction head)

Induction provably needs **≥ 2 attention layers**: one *previous-token* head to tag each position with
its predecessor, and one *induction* head to use that tag to find the earlier occurrence and copy what
came next. We keep it attention-only (no MLP) so the circuit stays legible. Build it and count its
parameters.

In [3]:
model = icl.InductionTransformer().to(icl.DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"layers: {icl.N_LAYERS} | d_model: {icl.D_MODEL} | heads: {icl.N_HEADS} "
      f"| head_dim: {icl.HEAD_DIM}")
print(f"total parameters: {n_params:,}  (tiny — trains on CPU in seconds)")

layers: 2 | d_model: 64 | heads: 4 | head_dim: 16
total parameters: 39,296  (tiny — trains on CPU in seconds)


## Train it on the recall task

We train with AdamW on fresh batches of the recall task, predicting only the final position (where
the target lives). Because each sequence's mapping is new, the model can't memorize answers — it can
only learn the *strategy* of looking back and copying. (We call the canonical `train_model()` so the
exact training matches the page.)

In [4]:
model = icl.train_model()  # deterministic: seeds inside, returns the trained model in eval mode
print(f"trained for {icl.N_TRAIN_STEPS} steps, batch size {icl.BATCH_SIZE}")
print("model is in eval mode:", not model.training)

trained for 1500 steps, batch size 128
model is in eval mode: True


## Experiment 1 — the induction head, caught looking back (the mechanism)

Now the payoff. We feed one recall sequence and read the **final layer's attention from the query
position**. An induction head should put almost all its weight on the position holding the value that
followed the query key's first occurrence — i.e. it *looks back and copies*. We assert the peak lands
exactly there.

In [5]:
look = icl.induction_lookback(model)
attn = torch.tensor(look["attn_from_query"])
vpos = look["lookback_value_pos"]
peak_pos = int(attn.argmax().item())

# The query position must place its STRONGEST attention on the value-after-the-key slot.
assert peak_pos == vpos, f"induction head did not look back (peak {peak_pos} != {vpos})"

print(f"query key id = {look['query_key']}, target value id = {look['target']}")
print(f"attention from the query peaks at position {peak_pos} "
      f"(the value after the key's first occurrence)")
print(f"weight on that look-back position: {attn[vpos].item():.2f}  -> it copies that value")
print("\nattention row from the query (rounded):")
print("  ", [round(x, 2) for x in look["attn_from_query"]])

query key id = 20, target value id = 16
attention from the query peaks at position 7 (the value after the key's first occurrence)
weight on that look-back position: 1.00  -> it copies that value

attention row from the query (rounded):
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## Experiment 2 — few-shot beats zero-shot (the core ICL claim)

We measure next-token accuracy as a function of `k`, the number of in-context demonstrations of the
queried rule. `k=0` (zero-shot) means the query key never appeared — nothing to copy, so the model is
at chance (~1/40 = 0.025). `k≥1` demonstrates the rule. We assert few-shot decisively beats zero-shot
**before** any timing — the result is the point, not the speed.

In [6]:
acc = icl.accuracy_vs_shots(model)

# The core ICL claim, asserted: demonstrations in the prompt raise accuracy, no weights changed.
assert acc[0] < 0.10, f"zero-shot should be ~chance, got {acc[0]:.3f}"
assert acc[1] > acc[0], "even one shot should beat zero-shot"
assert acc[max(icl.SHOT_VALUES)] > acc[0] + 0.3, "few-shot must decisively beat zero-shot"

print(f"{'k (shots)':>10} | accuracy")
print(f"{'-'*10}-+---------")
for k in icl.SHOT_VALUES:
    print(f"{k:>10} | {acc[k]:.3f}")
print(f"\nzero-shot = {acc[0]:.3f}  ->  {max(icl.SHOT_VALUES)}-shot = "
      f"{acc[max(icl.SHOT_VALUES)]:.3f}   (+{acc[max(icl.SHOT_VALUES)] - acc[0]:.3f})")
print("the task 'switches on' from a single demonstration -- with NO weight update.")

 k (shots) | accuracy
-----------+---------
         0 | 0.030
         1 | 1.000
         2 | 1.000
         3 | 1.000
         4 | 1.000
         5 | 1.000

zero-shot = 0.030  ->  5-shot = 1.000   (+0.970)
the task 'switches on' from a single demonstration -- with NO weight update.


## Experiment 3 — ICL is brittle (order sensitivity)

Real ICL is sensitive to choices unrelated to the task's logic. We probe this directly: show the
query key **twice with different values** (one says `→ A`, one says `→ B`), permute their order among
the distractors, and see which the model copies. A clean *first-occurrence* rule would always answer
`A`; any dependence on order is a recency/primacy **bias**. We measure how often the answer flips.

In [7]:
sens = icl.order_sensitivity(model)

# Brittleness, measured: a real fraction of sequences change answer purely from reordering.
assert sens["frac_order_sensitive"] > 0.05, "expected measurable order sensitivity"

print(f"fraction of sequences whose answer FLIPS with order : {sens['frac_order_sensitive']:.3f}")
print(f"recency rate (copies the LATER demo when it flips)  : {sens['recency_rate']:.3f}")
print("\nSame demonstrations, different order, different answer -> ICL is order-sensitive.")
print("(The fix, on real models, is CALIBRATION: probe with a content-free input to measure")
print(" the prompt's bias, then divide it out. See the page's calibration section.)")

fraction of sequences whose answer FLIPS with order : 0.570
recency rate (copies the LATER demo when it flips)  : 0.491

Same demonstrations, different order, different answer -> ICL is order-sensitive.
(The fix, on real models, is CALIBRATION: probe with a content-free input to measure
 the prompt's bias, then divide it out. See the page's calibration section.)


## Experiment 4 — content-free calibration removes label bias

Brittleness has a fix: **calibration** (Zhao et al. 2021). We flood a prompt with one majority label,
then probe it with a **content-free input** — a query key that never appears among the demonstrations,
so there is nothing to copy. A well-calibrated prompt would answer such an empty input *uniformly*;
any departure is pure prompt bias. The fix operates on **probabilities** (not logits): divide the
content-free distribution out and renormalise (the divisive special case of Zhao's affine
$\hat q = \mathrm{softmax}(\hat W \hat p + \hat b)$ with $\hat W = \mathrm{diag}(1/\hat p_{cf})$).
Applied to the content-free probe itself, it must flatten to uniform — the measured win we assert.

In [8]:
cal = icl.contextual_calibration(model)

# The calibration DIAGNOSTIC, measured: the content-free probe reveals bias, and dividing it
# out flattens the content-free distribution to the uniform prior.
assert cal["cf_majority_mass_before"] > 0.5, "biased prompt should pile mass on the majority label"
assert abs(cal["cf_majority_mass_after"] - cal["uniform_prior"]) < 1e-3, "calibration must flatten to uniform"

print(f"content-free probe, majority-label mass BEFORE : {cal['cf_majority_mass_before']:.3f}  "
      f"(uniform would be {cal['uniform_prior']:.3f})")
print(f"content-free probe, majority-label mass AFTER  : {cal['cf_majority_mass_after']:.3f}  "
      f"-> flattened to uniform (bias removed)")
print(f"downstream accuracy gain (Zhao 2021, illustrative): "
      f"{cal['reported_acc_before']:.3f} -> {cal['reported_acc_after']:.3f}")

content-free probe, majority-label mass BEFORE : 0.711  (uniform would be 0.025)
content-free probe, majority-label mass AFTER  : 0.025  -> flattened to uniform (bias removed)
downstream accuracy gain (Zhao 2021, illustrative): 0.616 -> 0.792


## Timing comes last — after every result is asserted

Only now, with all four qualitative results proven, do we time a forward pass — to show the model is
genuinely tiny. (We never let timing gate correctness.)

In [9]:
timing_tokens, timing_targets = icl.make_batch(
    icl.BATCH_SIZE, generator=torch.Generator().manual_seed(icl.SEED)
)
ms = icl.timeit(
    lambda: torch.nn.functional.cross_entropy(model(timing_tokens)[:, -1, :], timing_targets)
)
print(f"one forward+loss over a batch of {icl.BATCH_SIZE}: {ms:.1f} ms")

one forward+loss over a batch of 128: 4.6 ms


## Try it yourself

Before you run it, **predict**: in Experiment 3, if you made the model bigger (more layers/heads) and
trained it longer, would the order-sensitivity *fall*?

*Hint:* a cleaner, more consistent induction head would more reliably copy the **first** occurrence,
lowering the flip rate — brittleness is partly a symptom of an *imperfect* circuit. (This is also why
real, messy LLMs stay order-sensitive on hard prompts.) Try bumping `icl.N_TRAIN_STEPS` or
`icl.N_LAYERS` and re-running Experiment 3.

## What we saw

- **The induction head is real and visible.** A 2-layer model, trained only to predict the value after
  a repeated key, learned to put ~**1.00** of its attention on the look-back position — copying from
  context. This is the engine behind few-shot copying.
- **Few-shot beats zero-shot, with no weight update.** Accuracy jumped from **chance (~0.03)** to
  **1.00** with a single in-context demonstration of a rule the weights had never seen. The "learning"
  was entirely in the forward pass.
- **ICL is brittle.** With conflicting demonstrations, **~57%** of answers flipped purely from
  reordering — order, not just content, decides.
- **Calibration removes the bias.** A majority-label-biased prompt put **0.711** of its content-free
  probability on one label (uniform would be 0.025); dividing that bias out flattened it back to
  **0.025** — the defining property of a calibrated prompt. On real tasks this buys a sizable
  few-shot accuracy gain (Zhao et al. 2021).

Next: [Chain-of-Thought Reasoning](../../17-Chain-of-Thought-Reasoning/17-Chain-of-Thought-Reasoning.md)
— a *specific prompting technique* that makes the model emit reasoning steps before its answer.